# 57 â€” LGBM with PubChem PXR Bioassay Data

LGBM specifically trained with PubChem PXR bioassay data as additional training signal.

Key steps:
1. Load `data/external/pubchem_pxr_aids.parquet` (or fetch on-the-fly).
2. Data quality analysis â€” structural PAINS-like filters.
3. Weighting strategy: CRC=1.0, PubChem actives=0.4, PubChem inactives=0.2.
4. Ablation: (a) CRC only, (b) +actives only, (c) +inactives only, (d) +both.
5. Scaffold 5-fold CV + save.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles, to_inchikey, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')
train_inchikeys = set(tr['smiles'].map(to_inchikey).dropna())

Train: 4,139  Test: 513


## 1. Load PubChem PXR data

In [2]:
PUBCHEM_CACHE = DATA_EXTERNAL / 'pubchem_pxr_aids.parquet'

if PUBCHEM_CACHE.exists():
    print(f'Loading cached PubChem data from {PUBCHEM_CACHE}')
    pubchem_df = pd.read_parquet(PUBCHEM_CACHE)
    print(f'  {len(pubchem_df):,} rows loaded')
    if len(pubchem_df) == 0:
        print('Cached file is empty — will re-fetch from PubChem PUG REST')
        PUBCHEM_CACHE.unlink()  # delete so the else branch triggers
        pubchem_df = None  # sentinel
if PUBCHEM_CACHE.exists() and pubchem_df is not None and len(pubchem_df) > 0:
    pass  # already loaded above
else:
    print('pubchem_pxr_aids.parquet not found â€” fetching on-the-fly from PubChem PUG REST...')
    import requests, time, math

    PUBCHEM_BASE = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
    AID = 743219  # PXR reporter assay
    ACTIVE_PVAL = 6.5
    INACTIVE_PVAL = 3.0
    MAX_CIDS = 5000
    SMILES_BATCH = 100
    SLEEP = 0.3

    all_rows = []
    for act_type, pval in [('active', ACTIVE_PVAL), ('inactive', INACTIVE_PVAL)]:
        url = f'{PUBCHEM_BASE}/assay/aid/{AID}/cids/JSON?cids_type={act_type}'
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            cids = data.get('IdentifierList', {}).get('CID', [])[:MAX_CIDS]
            time.sleep(SLEEP)
            print(f'  AID {AID} {act_type}: {len(cids):,} CIDs')

            for i in range(0, len(cids), SMILES_BATCH):
                batch = cids[i:i+SMILES_BATCH]
                smi_url = (f'{PUBCHEM_BASE}/compound/cid/{",".join(map(str, batch))}'
                           f'/property/IsomericSMILES/JSON')
                try:
                    sr = requests.get(smi_url, timeout=30)
                    sr.raise_for_status()
                    for prop in sr.json().get('PropertyTable', {}).get('Properties', []):
                        smi = prop.get('IsomericSMILES', '')
                        if smi:
                            all_rows.append({'smiles': smi, 'activity': act_type, 'pec50': pval})
                except Exception:
                    pass
                time.sleep(SLEEP)
        except Exception as e:
            print(f'  Fetch failed for AID {AID} {act_type}: {e}')

    if all_rows:
        pubchem_df = pd.DataFrame(all_rows)
        pubchem_df['std_smiles'] = pubchem_df['smiles'].map(standardize_smiles)
        pubchem_df['inchikey'] = pubchem_df['std_smiles'].map(
            lambda s: to_inchikey(s) if s else None)
        pubchem_df = pubchem_df.dropna(subset=['std_smiles', 'inchikey']).reset_index(drop=True)
        pubchem_df = (
            pubchem_df.sort_values('pec50', ascending=False)
            .drop_duplicates(subset='inchikey', keep='first')
            .reset_index(drop=True)
        )
        pubchem_df.to_parquet(PUBCHEM_CACHE, index=False)
        print(f'Fetched and saved {len(pubchem_df):,} unique compounds')
    else:
        print('No data retrieved â€” creating empty placeholder')
        pubchem_df = pd.DataFrame(
            columns=['smiles', 'std_smiles', 'inchikey', 'activity', 'pec50'])
        pubchem_df.to_parquet(PUBCHEM_CACHE, index=False)

# Remove PXR training overlap
if 'inchikey' not in pubchem_df.columns:
    smi_col = next((c for c in ['std_smiles', 'smiles'] if c in pubchem_df.columns), None)
    if smi_col:
        pubchem_df['inchikey'] = pubchem_df[smi_col].map(to_inchikey)

before = len(pubchem_df)
pubchem_df = pubchem_df[~pubchem_df['inchikey'].isin(train_inchikeys)]
print(f'After removing PXR train overlap: {len(pubchem_df):,} (removed {before - len(pubchem_df):,})')

if 'activity' in pubchem_df.columns:
    print(pubchem_df['activity'].value_counts().to_string())

Loading cached PubChem data from D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\data\external\pubchem_pxr_aids.parquet
  0 rows loaded
After removing PXR train overlap: 0 (removed 0)
Series([], )


## 2. Data quality analysis â€” structural filters

In [3]:
if len(pubchem_df) > 0:
    smi_col = next((c for c in ['std_smiles', 'smiles'] if c in pubchem_df.columns), None)
    pubchem_df['smiles_use'] = pubchem_df[smi_col]

    # Compute physchem for filtering
    print('Computing physchem properties for filtering...')
    phys = pubchem_df['smiles_use'].map(compute_physchem)
    pubchem_df['mw']   = phys.map(lambda d: d['mw'] if d else np.nan)
    pubchem_df['logp'] = phys.map(lambda d: d['logp'] if d else np.nan)
    pubchem_df['tpsa'] = phys.map(lambda d: d['tpsa'] if d else np.nan)

    n_before = len(pubchem_df)

    # Apply PAINS-like / drug-likeness structural filters
    mask_mw   = (pubchem_df['mw']   >= 150) & (pubchem_df['mw']   <= 900)
    mask_tpsa = pubchem_df['tpsa']  <= 200
    mask_logp = (pubchem_df['logp'] >= -3)  & (pubchem_df['logp'] <= 10)
    mask_all  = mask_mw & mask_tpsa & mask_logp

    print(f'\nFilter statistics (n_before = {n_before:,}):')
    print(f'  MW [150, 900]:     {mask_mw.sum():,} pass ({mask_mw.mean()*100:.1f}%)')
    print(f'  TPSA <= 200:       {mask_tpsa.sum():,} pass ({mask_tpsa.mean()*100:.1f}%)')
    print(f'  logP [-3, 10]:     {mask_logp.sum():,} pass ({mask_logp.mean()*100:.1f}%)')
    print(f'  All filters:       {mask_all.sum():,} pass ({mask_all.mean()*100:.1f}%)')

    pubchem_filtered = pubchem_df[mask_all].reset_index(drop=True)
    print(f'\nCompounds after filtering: {len(pubchem_filtered):,}')

    if 'activity' in pubchem_filtered.columns:
        print(pubchem_filtered['activity'].value_counts().to_string())
    elif 'pec50' in pubchem_filtered.columns:
        print(pubchem_filtered['pec50'].describe().round(2))
else:
    pubchem_filtered = pd.DataFrame(columns=['smiles_use', 'pec50', 'activity'])
    print('PubChem data is empty â€” proceeding with CRC-only baseline.')

PubChem data is empty â€” proceeding with CRC-only baseline.


## 3. Featurize + weighting strategy

In [4]:
# â”€â”€ Featurize PXR training â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Featurizing PXR training set...')
X_tr = impute(combined(tr['smiles'].tolist()))
y_tr = tr['pec50'].values.astype(np.float32)
scaffolds_tr = tr['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds_tr, n_splits=N_FOLDS, seed=SEED)
print(f'X_tr: {X_tr.shape}')

# â”€â”€ Featurize test set â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('Featurizing test set...')
X_te = impute(combined(te['smiles'].tolist()))

# â”€â”€ Featurize PubChem compounds â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
X_pc = None
y_pc = None
w_pc_actives = None
w_pc_inactives = None

if len(pubchem_filtered) > 0 and 'smiles_use' in pubchem_filtered.columns:
    pec_col = 'pec50' if 'pec50' in pubchem_filtered.columns else None
    if pec_col is None:
        print('No pec50 column in PubChem data â€” skipping')
    else:
        print(f'Featurizing {len(pubchem_filtered):,} PubChem compounds...')
        X_pc = impute(combined(pubchem_filtered['smiles_use'].tolist()))
        y_pc = pubchem_filtered[pec_col].values.astype(np.float32)

        # Determine activity class
        if 'activity' in pubchem_filtered.columns:
            is_active = pubchem_filtered['activity'] == 'active'
        else:
            is_active = pubchem_filtered[pec_col] >= 5.0

        # Weight vectors
        w_pc_actives  = np.where(is_active, 0.4, 0.0).astype(np.float32)
        w_pc_inactives = np.where(~is_active, 0.2, 0.0).astype(np.float32)

        n_act = is_active.sum()
        n_inact = (~is_active).sum()
        print(f'PubChem: {n_act:,} actives (w=0.4), {n_inact:,} inactives (w=0.2)')
        print(f'X_pc: {X_pc.shape}')

print('Featurization complete.')

Featurizing PXR training set...


X_tr: (4139, 2265)
Featurizing test set...


Featurization complete.


## 4. Ablation study

In [5]:
def run_cv_with_pubchem(X_crc, y_crc, splits, X_aug=None, y_aug=None, w_aug=None,
                         lgbm_params=LGBM_PARAMS, label=''):
    """Scaffold CV adding optional augmented data to each fold's training split."""
    oof = np.full(len(y_crc), np.nan)
    w_crc = np.ones(len(y_crc), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(splits):
        Xf = X_crc[tr_idx]
        yf = y_crc[tr_idx]
        wf = w_crc[tr_idx]

        if X_aug is not None and len(X_aug) > 0:
            mask = w_aug > 0
            X_train = np.vstack([Xf, X_aug[mask]])
            y_train = np.concatenate([yf, y_aug[mask]])
            w_train = np.concatenate([wf, w_aug[mask]])
        else:
            X_train, y_train, w_train = Xf, yf, wf

        m = lgb.LGBMRegressor(**lgbm_params)
        m.fit(X_train, y_train, sample_weight=w_train, callbacks=[lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(X_crc[va_idx])

    return oof


ablation_results = {}

# (a) CRC only
print('(a) CRC only...')
oof_a = run_cv_with_pubchem(X_tr, y_tr, splits)
ablation_results['(a) CRC only'] = rae(y_tr, oof_a)
print(f'  OOF RAE = {ablation_results["(a) CRC only"]:.4f}')

if X_pc is not None:
    # (b) CRC + PubChem actives only
    print('(b) CRC + PubChem actives only...')
    oof_b = run_cv_with_pubchem(X_tr, y_tr, splits,
                                 X_aug=X_pc, y_aug=y_pc, w_aug=w_pc_actives)
    ablation_results['(b) +PubChem actives'] = rae(y_tr, oof_b)
    print(f'  OOF RAE = {ablation_results["(b) +PubChem actives"]:.4f}')

    # (c) CRC + PubChem inactives only
    print('(c) CRC + PubChem inactives only...')
    oof_c = run_cv_with_pubchem(X_tr, y_tr, splits,
                                 X_aug=X_pc, y_aug=y_pc, w_aug=w_pc_inactives)
    ablation_results['(c) +PubChem inactives'] = rae(y_tr, oof_c)
    print(f'  OOF RAE = {ablation_results["(c) +PubChem inactives"]:.4f}')

    # (d) CRC + both
    print('(d) CRC + PubChem both...')
    w_both = (w_pc_actives + w_pc_inactives)  # one per compound, since they are 0/nonzero
    oof_d = run_cv_with_pubchem(X_tr, y_tr, splits,
                                 X_aug=X_pc, y_aug=y_pc, w_aug=w_both)
    ablation_results['(d) +both'] = rae(y_tr, oof_d)
    print(f'  OOF RAE = {ablation_results["(d) +both"]:.4f}')
else:
    print('No PubChem data available â€” only CRC baseline computed.')

print('\n=== Ablation Summary ===')
ablation_df = pd.DataFrame(
    [{'config': k, 'OOF RAE': v} for k, v in ablation_results.items()]
).sort_values('OOF RAE')
print(ablation_df.to_string(index=False))

# Select best configuration
best_config = ablation_df.iloc[0]['config']
best_oof_map = {'(a) CRC only': oof_a}
if X_pc is not None:
    best_oof_map.update({'(b) +PubChem actives': oof_b,
                          '(c) +PubChem inactives': oof_c,
                          '(d) +both': oof_d})
best_oof = best_oof_map[best_config]
print(f'\nBest config: {best_config}  OOF RAE = {rae(y_tr, best_oof):.4f}')

(a) CRC only...


  OOF RAE = 0.5600
No PubChem data available â€” only CRC baseline computed.

=== Ablation Summary ===
      config  OOF RAE
(a) CRC only 0.560014

Best config: (a) CRC only  OOF RAE = 0.5600


## 5. Final model + save

In [6]:
# Build training data for final model using best configuration
w_base = np.ones(len(y_tr), dtype=np.float32)

if best_config == '(a) CRC only' or X_pc is None:
    X_final, y_final, w_final = X_tr, y_tr, w_base
elif best_config == '(b) +PubChem actives':
    mask = w_pc_actives > 0
    X_final = np.vstack([X_tr, X_pc[mask]])
    y_final = np.concatenate([y_tr, y_pc[mask]])
    w_final = np.concatenate([w_base, w_pc_actives[mask]])
elif best_config == '(c) +PubChem inactives':
    mask = w_pc_inactives > 0
    X_final = np.vstack([X_tr, X_pc[mask]])
    y_final = np.concatenate([y_tr, y_pc[mask]])
    w_final = np.concatenate([w_base, w_pc_inactives[mask]])
else:  # both
    w_both_final = w_pc_actives + w_pc_inactives
    mask = w_both_final > 0
    X_final = np.vstack([X_tr, X_pc[mask]])
    y_final = np.concatenate([y_tr, y_pc[mask]])
    w_final = np.concatenate([w_base, w_both_final[mask]])

print(f'Training final model on {len(y_final):,} compounds...')
final_model = lgb.LGBMRegressor(**LGBM_PARAMS)
final_model.fit(X_final, y_final, sample_weight=w_final,
                callbacks=[lgb.log_evaluation(-1)])

te_preds = np.clip(final_model.predict(X_te),
                   float(y_tr.min()) - 0.5,
                   float(y_tr.max()) + 0.5)

print(f'Test preds â€” mean={te_preds.mean():.3f}  std={te_preds.std():.3f}')

# Save
np.save(DATA_PROCESSED / 'oof_lgbm_pubchem.npy', best_oof)
np.save(DATA_PROCESSED / 'te_lgbm_pubchem.npy', te_preds)
print(f'Saved oof_lgbm_pubchem.npy  OOF RAE = {rae(y_tr, best_oof):.4f}')

sub = pd.DataFrame({'Molecule Name': te['name'].values, 'pEC50': te_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '57_lgbm_pubchem_pxr.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

print('\n== Summary ==')
print(ablation_df.to_string(index=False))
print(f'\nBest config: {best_config}')
sub['pEC50'].describe().round(3)

Training final model on 4,139 compounds...


Test preds â€” mean=4.804  std=0.649
Saved oof_lgbm_pubchem.npy  OOF RAE = 0.5600
Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\57_lgbm_pubchem_pxr.csv

== Summary ==
      config  OOF RAE
(a) CRC only 0.560014

Best config: (a) CRC only


count    513.000
mean       4.804
std        0.650
min        2.209
25%        4.443
50%        4.948
75%        5.282
max        6.052
Name: pEC50, dtype: float64